# 📊 Power Outage Dataset Cleaning & Preprocessing
This notebook performs cleaning, preprocessing, and feature engineering on:
- `eaglei_outages_2019.csv`
- `MCC.csv`
- `DQI.csv`
- `coverage_history.csv`

It also calculates DOI (Duration of Interruption) by grouping related outage events.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Load the datasets
outages = pd.read_csv("eaglei_outages_2019.csv")
mcc = pd.read_csv("MCC.csv")
dqi = pd.read_csv("DQI.csv")
coverage = pd.read_csv("coverage_history.csv")

# Filter for Michigan only
outages = outages[outages["state"] == "Michigan"].copy()

# Clean and format datetime
outages['run_start_time'] = pd.to_datetime(outages['run_start_time'], format='ISO8601')

# Ensure FIPS is string with zero padding
outages['fips_code'] = outages['fips_code'].astype(str).str.zfill(5)
mcc['County_FIPS'] = mcc['County_FIPS'].astype(str).str.zfill(5)

# Merge with MCC to get customer population
outages = outages.merge(mcc, left_on='fips_code', right_on='County_FIPS', how='left')

# Calculate outage rate
outages['outage_rate'] = outages['customers_out'] / outages['Customers']
outages = outages.drop(columns=['County_FIPS'])

# Extract date and time components early
outages['start_date'] = outages['run_start_time'].dt.date.astype(str)
outages['start_time'] = outages['run_start_time'].dt.time.astype(str)

print("Basic preprocessing complete. Starting improved event detection...")

# --- IMPROVED EVENT DETECTION WITH REALISTIC ENHANCEMENTS ---
print("Applying enhanced logic to define outage events...")
outages = outages.sort_values(['fips_code', 'run_start_time'])
outages['time_diff'] = outages.groupby('fips_code')['run_start_time'].diff()

# Track changes in customer outages
outages['customers_diff'] = outages.groupby('fips_code')['customers_out'].diff().fillna(0)

# Get county capacity for adaptive thresholds
outages['county_capacity'] = outages.groupby('fips_code')['Customers'].transform('first')
outages['customers_pct_change'] = outages['customers_diff'] / outages['county_capacity']

# Detect recovery phases (significant drops in customers out)
is_recovery = outages['customers_diff'] < -1000
outages['is_recovery_phase'] = is_recovery

# Enhanced event triggers with adaptive thresholds
is_first_event = outages['time_diff'].isna()
is_long_gap = outages['time_diff'] > pd.Timedelta(hours=3)

# Adaptive threshold: 20% increase OR absolute 5000, whichever is smaller for the county
adaptive_threshold = np.minimum(0.20 * outages['county_capacity'], 5000)
is_new_major_outage = (outages['customers_diff'] > adaptive_threshold) & ~is_recovery

# Additional trigger: Very long gaps (12+ hours) even with smaller increases
is_very_long_gap = outages['time_diff'] > pd.Timedelta(hours=12)

# Combine all triggers
outages['new_event'] = is_first_event | is_long_gap | is_new_major_outage | is_very_long_gap
outages['event_id'] = outages.groupby('fips_code')['new_event'].cumsum()

print("Enhanced event IDs calculated.")

# --- COMPUTE DOI WITH VALIDATION ---
print("Calculating DOI for each event with validation...")
doi_df = outages.groupby(['fips_code', 'event_id']).agg(
    start_time=('run_start_time', 'min'),
    end_time=('run_start_time', 'max'),
    max_customers=('customers_out', 'max'),
    max_rate=('outage_rate', 'max'),
    report_count=('run_start_time', 'count')
).reset_index()

# Calculate DOI with realistic limits
doi_df['DOI_hours'] = (doi_df['end_time'] - doi_df['start_time']) / pd.Timedelta(hours=1)
doi_df['DOI_hours'] = doi_df['DOI_hours'].clip(upper=168)  # Max 7 days

# Validate events
doi_df['is_valid_event'] = (
    (doi_df['DOI_hours'] >= 0.5) &  # Minimum 30 minutes
    (doi_df['report_count'] >= 1) &  # At least 1 report
    (doi_df['max_customers'] > 0)   # Must have customers affected
)

# Classify event severity
def classify_event_severity(row):
    max_rate = row['max_rate']
    max_customers = row['max_customers']
    duration = row['DOI_hours']
    
    if max_rate > 0.5 or max_customers > 50000 or duration > 48:
        return 'Major'
    elif max_rate > 0.2 or max_customers > 10000 or duration > 12:
        return 'Moderate'
    else:
        return 'Minor'

doi_df['event_severity'] = doi_df.apply(classify_event_severity, axis=1)

print("DOI calculation complete with validation.")

# --- MERGE AND CLEANUP ---
print("Merging enhanced DOI data back to main dataframe...")
outages = outages.merge(
    doi_df[['fips_code', 'event_id', 'DOI_hours', 'event_severity', 'is_valid_event']], 
    on=['fips_code', 'event_id'], 
    how='left'
)

# Filter out invalid events if desired (optional)
print(f"Total events: {len(doi_df)}")
print(f"Valid events: {doi_df['is_valid_event'].sum()}")
print(f"Invalid events removed: {(~doi_df['is_valid_event']).sum()}")

# Clean up temporary columns
columns_to_drop = [
    'time_diff', 'customers_diff', 'new_event', 'county_capacity', 
    'customers_pct_change', 'is_recovery_phase'
]
outages = outages.drop(columns=[col for col in columns_to_drop if col in outages.columns])

print("\nEvent classification summary:")
print(outages['event_severity'].value_counts())

# --- ADD COORDINATES ---
print("Adding geographic coordinates...")
fips_coords = pd.read_csv("https://gist.githubusercontent.com/russellsamora/12be4f9f574e92413ea3f92ce1bc58e6/raw/us_county_latlng.csv")
fips_coords['fips_code'] = fips_coords['fips_code'].astype(str).str.zfill(5)

# Merge coordinates
michigan_outages = outages.merge(
    fips_coords[['fips_code', 'lat', 'lng']],
    on='fips_code',
    how='left'
)

# --- FINAL VALIDATION AND CLEANUP ---
print("Performing final validation...")

# Check for missing coordinates
missing_coords = michigan_outages[['lat', 'lng']].isnull().any(axis=1).sum()
if missing_coords > 0:
    print(f"⚠️ Warning: {missing_coords} rows missing coordinates")

# Rename columns for clarity
michigan_outages = michigan_outages.rename(columns={
    'start_date': 'start_date_only',
    'start_time': 'start_time_only'
})

# Remove duplicate run_start_time column if it exists
if 'run_start_datetime' in michigan_outages.columns:
    michigan_outages = michigan_outages.drop(columns=['run_start_datetime'])

# --- SELECT ONLY SPECIFIED COLUMNS ---
print("Filtering to keep only specified columns...")
columns_to_keep = [
    'fips_code', 'county', 'state', 'customers_out', 'Customers', 
    'outage_rate', 'event_id', 'start_date_only', 'start_time_only', 
    'DOI_hours', 'lat', 'lng'
]

# Filter the dataframe to keep only these columns
michigan_outages_filtered = michigan_outages[columns_to_keep]

# --- SAVE RESULTS ---
michigan_outages_filtered.to_csv("new_michigan_outages_2019.csv", index=False)

print("✅ Enhanced preprocessing complete!")
print(f"Final dataset shape: {michigan_outages_filtered.shape}")
print(f"Columns in final dataset: {list(michigan_outages_filtered.columns)}")
print(f"Unique counties: {michigan_outages_filtered['fips_code'].nunique()}")
print(f"Total events: {michigan_outages_filtered['event_id'].max()}")

# Display sample of final results
print("\nSample of filtered dataset:")
print(michigan_outages_filtered.head(10))

# --- OPTIONAL: SAVE EVENT SUMMARY ---
event_summary = doi_df[doi_df['is_valid_event']].copy()
event_summary.to_csv("new_michigan_outage_events_summary_2019.csv", index=False)
print(f"\n✅ Event summary saved to 'michigan_outage_events_summary_2019.csv'")
print(f"Valid events saved: {len(event_summary)}")

In [ ]:
michigan_outages_filtered.head()

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("michigan_outages_2019.csv")

# Drop all rows with any null values
df_clean = df.dropna()

print(f"Rows after dropping nulls: {len(df_clean)} (removed {len(df) - len(df_clean)} rows)")

# Optionally, save the cleaned DataFrame
df_clean.to_csv("michigan_outages_2019_clean.csv", index=False)

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("michigan_outages_2019_clean.csv")

# Show basic info
print("=== DataFrame Info ===")
df.info()

# Show number of nulls per column
print("\n=== Null Values Per Column ===")
print(df.isnull().sum())

# Show total number of rows with any null value
num_rows_with_nulls = df.isnull().any(axis=1).sum()
print(f"\nRows with at least one null value: {num_rows_with_nulls} / {len(df)}")

# Show percentage of nulls per column
print("\n=== Percentage of Nulls Per Column ===")
print((df.isnull().mean() * 100).round(2))

# Show a sample of rows with nulls (if any)
if num_rows_with_nulls > 0:
    print("\n=== Sample rows with nulls ===")
    display(df[df.isnull().any(axis=1)].head())
else:
    print("\nNo rows with null values.")

# Show basic statistics
print("\n=== Basic Statistics ===")
display(df.describe(include='all'))

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("new_michigan_outages_2019.csv")

# Keep only unique lat/lng pairs
unique_lat_lng = df[['lat', 'lng']].drop_duplicates()

# Save to CSV
unique_lat_lng.to_csv("unique_lat_lng.csv", index=False)
print(f"Saved {len(unique_lat_lng)} unique lat/lng pairs to 'unique_lat_lng.csv'")

In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from tqdm import tqdm

# Load your unique lat/lon pairs
stations = pd.read_csv("unique_lat_lng.csv")
# Define which hourly variables you want
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",              # Optional but useful in winter outage modeling
    "cloud_cover",
    "surface_pressure",        # Important for detecting storm fronts
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",          # Critical for capturing damaging winds
    "weather_code"             # Optional, categorical weather descriptor
]


def fetch_open_meteo_hourly(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARS),
        "timezone": "auto"
    }
    resp = requests.get(url, params=params, timeout=60)
    if resp.status_code == 200:
        return resp.json()
    else:
        print(f"API error {resp.status_code}: {resp.text[:100]}")
        return None

all_records = []
# Python
for idx, row in tqdm(stations.iterrows(), total=len(stations)):
    lat, lon = row['lat'], row['lng']
    print(f"Processing station {idx+1}/{len(stations)} at lat={lat}, lon={lon}...")
    data = fetch_open_meteo_hourly(lat, lon, "2019-01-01", "2019-12-31")
    if data and "hourly" in data and "time" in data["hourly"]:
        n = len(data["hourly"]["time"])
        print(f"  ✔️  Retrieved {n} hourly records for {lat},{lon}")
        for i in range(n):
            record = {
                "lat": lat,
                "lng": lon,
                "datetime": data["hourly"]["time"][i]
            }
            for var in HOURLY_VARS:
                record[var] = data["hourly"].get(var, [None]*n)[i]
            all_records.append(record)
    else:
        print(f"  ⚠️ No hourly data for {lat},{lon} in 2019")
    time.sleep(1)  # Respect rate limits

hourly_df = pd.DataFrame(all_records)
hourly_df.to_csv("hourly_weather_2019_all_stations.csv", index=False)
print("✅ Hourly weather data for 2019 saved to 'hourly_weather_2019_all_stations.csv'")

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("hourly_weather_2019_all_stations.csv")

# Show basic info
print("=== DataFrame Info ===")
df.info()

# Show number of nulls per column
print("\n=== Null Values Per Column ===")
print(df.isnull().sum())

# Show total number of rows with any null value
num_rows_with_nulls = df.isnull().any(axis=1).sum()
print(f"\nRows with at least one null value: {num_rows_with_nulls} / {len(df)}")

# Show percentage of nulls per column
print("\n=== Percentage of Nulls Per Column ===")
print((df.isnull().mean() * 100).round(2))

# Show a sample of rows with nulls (if any)
if num_rows_with_nulls > 0:
    print("\n=== Sample rows with nulls ===")
    print(df[df.isnull().any(axis=1)].head())
else:
    print("\nNo rows with null values.")

# Check for empty strings ("" or " ") in string/object columns
object_cols = df.select_dtypes(include=['object']).columns
empty_str_mask = (df[object_cols] == "") | (df[object_cols] == " ")
rows_with_empty_str = df[empty_str_mask.any(axis=1)]
print(f"\nRows with empty string values: {len(rows_with_empty_str)}")
if len(rows_with_empty_str) > 0:
    print(rows_with_empty_str.head())

In [ ]:
import pandas as pd

# Load outage data and weather data
outages = pd.read_csv("new_michigan_outages_2019.csv")
weather = pd.read_csv("hourly_weather_2019_all_stations.csv")

# Parse start_date_only and start_time_only as datetime
outages['datetime_full'] = pd.to_datetime(
    outages['start_date_only'] + ' ' + outages['start_time_only'],
    errors='coerce'
)

# Round to the nearest hour (floor if <30min, ceil if >=30min)
outages['datetime_hour'] = outages['datetime_full'].dt.round('H')
outages['datetime'] = outages['datetime_hour'].dt.strftime('%Y-%m-%dT%H:00')

# Format as 'YYYY-MM-DDTHH:00'
outages['datetime'] = outages['datetime_hour'].dt.strftime('%Y-%m-%dT%H:00')

# Round lat/lng to match precision (important for merge)
outages['lat'] = outages['lat'].round(8)
outages['lng'] = outages['lng'].round(8)
weather['lat'] = weather['lat'].round(8)
weather['lng'] = weather['lng'].round(8)

# Merge on lat, lng, and datetime
merged = outages.merge(
    weather,
    on=['lat', 'lng', 'datetime'],
    how='left',
    suffixes=('', '_weather')
)

# Save the merged DataFrame
merged.to_csv("new_michigan_outages_2019_with_weather.csv", index=False)
print("✅ Weather data merged and saved to michigan_outages_2019_with_weather.csv")

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("michigan_outages_2019_with_weather.csv")

# Show basic info
print("=== DataFrame Info ===")
df.info()

# Show number of nulls per column
print("\n=== Null Values Per Column ===")
print(df.isnull().sum())

# Show total number of rows with any null value
num_rows_with_nulls = df.isnull().any(axis=1).sum()
print(f"\nRows with at least one null value: {num_rows_with_nulls} / {len(df)}")

# Show percentage of nulls per column
print("\n=== Percentage of Nulls Per Column ===")
print((df.isnull().mean() * 100).round(2))

# Show a sample of rows with nulls (if any)
if num_rows_with_nulls > 0:
    print("\n=== Sample rows with nulls ===")
    print(df[df.isnull().any(axis=1)].head())
else:
    print("\nNo rows with null values.")

# Check for empty strings ("" or " ") in string/object columns
object_cols = df.select_dtypes(include=['object']).columns
empty_str_mask = (df[object_cols] == "") | (df[object_cols] == " ")
rows_with_empty_str = df[empty_str_mask.any(axis=1)]
print(f"\nRows with empty string values: {len(rows_with_empty_str)}")
if len(rows_with_empty_str) > 0:
    print(rows_with_empty_str.head())

In [ ]:
df.head()

In [ ]:
import pandas as pd

# Load the merged dataset
df = pd.read_csv("michigan_outages_2019_with_weather.csv")

# Select only the necessary columns for modeling and analysis
columns_to_keep = [
    'datetime', 'lat', 'lng', 'fips_code', 'county', 'customers_out', 'outage_rate', 'DOI_hours',
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature',
    'precipitation', 'rain', 'snowfall', 'cloud_cover', 'wind_speed_10m', 'wind_direction_10m'
]

df_reduced = df[columns_to_keep]

# Save the reduced dataset
df_reduced.to_csv("michigan_outages_2019_for_modeling.csv", index=False)
print("✅ Saved reduced dataset as 'michigan_outages_2019_for_modeling.csv'")

In [ ]:
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from tqdm import tqdm

# Load your unique lat/lon pairs
stations = pd.read_csv("unique_lat_lng.csv")
# Define which hourly variables you want
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",              # Optional but useful in winter outage modeling
    "cloud_cover",
    "surface_pressure",        # Important for detecting storm fronts
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",          # Critical for capturing damaging winds
    "weather_code"             # Optional, categorical weather descriptor
]


def fetch_open_meteo_hourly(lat, lon, start_date, end_date):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(HOURLY_VARS),
        "timezone": "auto"
    }
    resp = requests.get(url, params=params, timeout=60)
    if resp.status_code == 200:
        return resp.json()
    else:
        print(f"API error {resp.status_code}: {resp.text[:100]}")
        return None

all_records = []
# Python
for idx, row in tqdm(stations.iterrows(), total=len(stations)):
    lat, lon = row['lat'], row['lng']
    print(f"Processing station {idx+1}/{len(stations)} at lat={lat}, lon={lon}...")
    data = fetch_open_meteo_hourly(lat, lon, "2019-01-01", "2019-12-31")
    if data and "hourly" in data and "time" in data["hourly"]:
        n = len(data["hourly"]["time"])
        print(f"  ✔️  Retrieved {n} hourly records for {lat},{lon}")
        for i in range(n):
            record = {
                "lat": lat,
                "lng": lon,
                "datetime": data["hourly"]["time"][i]
            }
            for var in HOURLY_VARS:
                record[var] = data["hourly"].get(var, [None]*n)[i]
            all_records.append(record)
    else:
        print(f"  ⚠️ No hourly data for {lat},{lon} in 2019")
    time.sleep(1)  # Respect rate limits

hourly_df = pd.DataFrame(all_records)
hourly_df.to_csv("hourly_weather_2019_all_stations.csv", index=False)
print("✅ Hourly weather data for 2019 saved to 'hourly_weather_2019_all_stations.csv'")

In [ ]:
import requests
import pandas as pd

# County coordinates
county_coords = {
    'wayne': (42.28, -83.26),
    'oakland': (42.66, -83.38),
    'macomb': (42.71, -82.82),
    'monroe': (41.73, -83.45),
    'washtenaw': (42.25, -83.84)
}

# Weather variables to fetch
weather_vars = [
    "temperature_2m",
    "precipitation",
    "relative_humidity_2m",
    "windspeed_10m",
    "windgusts_10m",
    "winddirection_10m",
    "surface_pressure",
    "cloudcover",
    "shortwave_radiation",
    "dewpoint_2m"
]

# Date parameters
start_date = "2020-01-01"
end_date = "2024-12-31"

# Store all results
all_data = []

# Fetch data per county
for county, (lat, lon) in county_coords.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(weather_vars),
        "timezone": "auto"
    }

    response = requests.get("https://archive-api.open-meteo.com/v1/archive", params=params)
    data = response.json()

    if "hourly" in data:
        df = pd.DataFrame(data["hourly"])
        df['DTTM'] = pd.to_datetime(df['time'])
        df.drop(columns='time', inplace=True)
        df['county'] = county
        df['LATITUDE'] = lat
        df['LONGITUDE'] = lon
        all_data.append(df)

# Combine all counties
final_df = pd.concat(all_data, ignore_index=True)

# Save to CSV
final_df.to_csv("five_counties_weather.csv", index=False)
print("Saved to five_counties_weather.csv")
